# 🌀 turnover_momentum — short-term momentum, only where the tape is deep

A **daily, cross-sectional** strategy on price and volume alone. Sorting stocks on last
month's return *and* on share turnover splits the classic one-month effect in two: thinly
traded names **reverse**, heavily traded names **continue** (Medhat & Schmeling, RFS 2022 —
"as profitable and as persistent as conventional price momentum", "survives transaction
costs", "strongest among the largest, most liquid" stocks). This notebook trades the
continuation leg only:

- **Eligibility** — turnover rank ≥ `turnover_q` (the more-traded half by default); every
  other name is dropped from the cross-section. `turnover_q: 0.0` is the plain-momentum
  control the paper says should be weaker or reversed.
- **Score** — the trailing `formation`-bar return, ending `skip` bars ago.
- **Book** — the top `max_positions` names, one fixed slot each, re-picked every
  `rebalance_days` bars and carried in between; one shared $10k book (portfolio mode),
  next-bar-open fills, `retrim_k` caps a runaway winner.

Universe: `Liquid500`, 500 names ranked **point-in-time** by dollar volume over the half-year
*before* the train window (`notebooks/01_universe_screener.ipynb` §8b). Train 2020-07 →
2024-12; 2025+ is the untouched holdout.

Flow: **load → overview → backtest + baseline → §4 pre-registered signal test →
falsification sweep → optimize → pick → boundary → validate (hurdle, cost stress,
sub-period, holdout).**

> **§4 is the gate, and it is pre-registered.** Positive demeaned rank-IC in the *top*
> turnover tercile, lower or negative in the *bottom* one, on the train window. If it
> fails, stop there and do not run the study.

> Shares outstanding come from the fundamentals sleeve (`fund_shares`). If the coverage
> printed in §1 reads low, run `notebooks/03_altdata_update.ipynb` with the fundamentals
> dataset for `Liquid500` first; the strategy falls back to dollar volume otherwise.

## 1 · Load the panel and attach shares outstanding

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, features, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("turnover_momentum")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}")
for name, frame in panel.features.items():
    covered = int(frame.notna().any().sum())
    print(f"  feature {name:12s}: {covered}/{len(syms)} symbols with data")

## 2 · Strategy overview

The parameter table below is rendered straight from `config.yaml`, so it never
drifts from what actually runs.

In [ ]:
print("defaults (params):")
for k, v in strat.params.items():
    print(f"  {k:16s} = {v}")
print("\nsweep grid:")
for k, v in strat.sweep.items():
    print(f"  {k:16s} : {v}")
print("\noptuna search space:")
for k, v in strat.optuna["space"].items():
    print(f"  {k:16s} : {v}")

## 3 · One backtest at the defaults — and the baseline it has to beat

Costs are always on (commission + half-spread + slippage per leg). The second figure is the
**mandatory baseline comparison**: equal-weight buy-and-hold of the same 500 names over the
same window, paying the same costs — `excess_cagr` and `info_ratio` are measured against
it. Read `pct_profitable` (share of traded names that made money) as the breadth check.

In [ ]:
res = bt.run_strategy(strat, panel, settings=settings)
stats0 = bt.portfolio_stats(res)
fees0, slip0 = bt.panel_costs(settings.costs, panel)     # match run_strategy's pricing
stats0.update(bt.active_stats(res, panel, fees=fees0, slippage=slip0,
                              fill=strat.fill, total_return=stats0["total_return"]))
for k, v in stats0.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res)).show()

# The baseline: strategies/baseline (== backtest.benchmark, test-pinned) on the SAME panel.
import plotly.graph_objects as go
INIT_CASH = settings.backtest.init_cash
res_base = bt.run_strategy(strategies.load("baseline"), panel, settings=settings)
fig = go.Figure()
fig.add_trace(plots._scatter(
    bt.equity_curve(res_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(
    bt.equity_curve(res, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="turnover_momentum", line={"color": plots.SERIES[0], "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>turnover_momentum</extra>"))
plots._base(fig, title=f"{d['universe']} · train — cumulative return vs equal-weight buy-and-hold",
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 4 · The pre-registered test — does the effect exist on this universe at all?

**Run this before anything else, and decide on it.** The thesis is not "momentum works";
it is "one-month momentum *continues* where turnover is high and *reverses* where it is
low". That is a statement about the cross-section that needs no trading engine, so it is
tested here without one — the way `gap_and_go` should have been tested before its study.

Method: rank IC of the formation return against **cross-sectionally demeaned** forward
returns (subtracting each day's mean return removes the market, leaving stock selection),
computed separately inside each **turnover tercile**. The t-stat is divided by √horizon
because overlapping forward windows make consecutive daily ICs dependent.

**Pass** = the 21-day IC is positive in the *top* tercile and lower (ideally negative) in
the *bottom* one. **Fail** = stop here. Do not run §6; write the result into `IDEAS.md`.

The last table scores the ranking the strategy actually trades (eligible names only) — the
number §10a has to harvest after costs.

In [ ]:
HORIZONS = (5, 21, 42)            # ~1 week, the 1-month holding period, ~2 months
scores = strat.signals.__globals__["scores"]   # public helper on the strategy module

def signal_quality(score: pd.DataFrame, close: pd.DataFrame) -> pd.DataFrame:
    """Rank IC and decile spread of a score against DEMEANED forward returns.

    Demeaning each day's forward returns cross-sectionally removes the market, so what
    remains measures stock selection alone. The t-stat is divided by sqrt(horizon)
    because overlapping forward windows make consecutive daily ICs highly dependent —
    the naive t-stat is inflated by ~sqrt(h).
    """
    rows = []
    for h in HORIZONS:
        fwd = close.shift(-h) / close - 1.0
        rel = fwd.sub(fwd.mean(axis=1), axis=0)          # <- the market, removed
        ic = score.corrwith(rel, axis=1, method="spearman").dropna()
        naive_t = ic.mean() / (ic.std() / np.sqrt(len(ic))) if len(ic) and ic.std() else 0.0
        r = score.rank(axis=1, pct=True)
        rows.append({
            "horizon": f"{h}d",
            "rank_IC": round(float(ic.mean()), 4),
            "t_stat (overlap-adj)": round(float(naive_t / np.sqrt(h)), 2),
            "top_decile": float(rel.where(r >= 0.9).mean(axis=1).mean()),
            "bottom_decile": float(rel.where(r <= 0.1).mean(axis=1).mean()),
        })
    out = pd.DataFrame(rows).set_index("horizon")
    out["spread"] = out["top_decile"] - out["bottom_decile"]
    return out

FMT = {"top_decile": "{:+.2%}", "bottom_decile": "{:+.2%}", "spread": "{:+.2%}"}
parts_q = scores(panel, **strat.params)
print(f"turnover source: {parts_q['turnover_source']}"
      "  (shares = volume / fund_shares in split-consistent units; dollar_volume = fallback)")
print(f"eligible names per day (median): {int(parts_q['eligible'].sum(axis=1).median())}")

ret, turn = parts_q["ret"], parts_q["turnover_rank"]
terciles = {"top (most traded)": turn > 2 / 3,
            "middle": (turn > 1 / 3) & (turn <= 2 / 3),
            "bottom (least traded)": turn <= 1 / 3}
print("\nTHE PRE-REGISTERED TEST — formation return vs demeaned forward return,"
      " by turnover tercile")
rows = {}
for name, mask in terciles.items():
    q = signal_quality(ret.where(mask), panel.close)
    rows[name] = {**{f"{h} IC": q.loc[h, "rank_IC"] for h in q.index},
                  **{f"{h} t": q.loc[h, "t_stat (overlap-adj)"] for h in q.index},
                  **{f"{h} spread": q.loc[h, "spread"] for h in q.index}}
table = pd.DataFrame(rows).T
display(table.style.format({c: "{:+.2%}" for c in table.columns if c.endswith("spread")}
                           | {c: "{:+.4f}" for c in table.columns if c.endswith("IC")}
                           | {c: "{:+.2f}" for c in table.columns if c.endswith(" t")}))
top_ic = table.loc["top (most traded)", "21d IC"]
bot_ic = table.loc["bottom (least traded)", "21d IC"]
verdict = "PASS" if (top_ic > 0 and top_ic > bot_ic) else "FAIL"
print(f"\n21-day IC: top tercile {top_ic:+.4f}  vs  bottom tercile {bot_ic:+.4f}  -->  {verdict}")
print("PASS = positive where turnover is high AND lower where it is low."
      " FAIL = stop; do not run §6.")

print("\nTHE RANKING THE STRATEGY ACTUALLY TRADES (eligible names only, signed by polarity)")
display(signal_quality(parts_q["score_pct"], panel.close).style.format(FMT))

## 5 · Coarse sweep — the falsification grid

Not a tuning run: the grid crosses the thesis controls — `turnover_q` `[0.0, 0.5, 0.8]`
(no conditioning / the more-traded half / the top quintile), `polarity`
`[momentum, reversal]`, `skip` `[0, 5]` and `max_positions` `[10, 20]` — with the signal
window pinned at the shipped month. The paper predicts `excess_cagr` rises with
`turnover_q` under `momentum` and falls under `reversal`. If `turnover_q: 0.0` wins, the
conditioning adds nothing here; if `reversal` wins on the high-turnover slice, the thesis
is inverted on this universe.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
sweep = bt.sweep(strat, panel, grid, settings=settings)
display(sweep.sort_values("excess_cagr", ascending=False).head(10).round(3))
print("excess_cagr by turnover floor × polarity (skip and slot count averaged out):")
display(sweep.pivot_table(index="turnover_q", columns="polarity", values="excess_cagr",
                          aggfunc="mean").round(3))
plots.plot_sweep(sweep).show()

## 6 · Optimize (Optuna, resumable) — only after §4 passed

Scores `excess_cagr` (maximized) against `tracking_error` (minimized) — a Pareto
front, both measured relative to equal-weight buy-and-hold, so excess is above zero
only by beating the market; the pick only chooses front trials that do. Five searched knobs (`formation`, `skip`, `turnover_window`, `turnover_q`,
`rebalance_days`, plus the `polarity` control); `max_positions` is deliberately **not**
searched — it is a risk decision, and informed_flow's studies picked its floor six times
running. The study is journal-backed, so this cell resumes if re-run.

In [ ]:
N_OPTIMIZE_TRIALS = 400
jobs.run_optimize("turnover_momentum", n_trials=N_OPTIMIZE_TRIALS)

## 7 · Read the study — admissibility before picking

Few-trade parameter sets can post a flattering number by luck, so we filter the
Pareto front to trials with at least `MIN_TRADES` before choosing. Among the
admissible front we take the objective-weighted best, picking on the study's own
benchmark-relative objectives (never absolute Sharpe, which would just select the
most market-exposed combo).

In [ ]:
MIN_TRADES = 100    # ~24 trades/yr x 4.5 train years
study = opt.load_study(strat)
front = opt.pareto_frame(study)
admissible = front[front["n_trades"] >= MIN_TRADES].copy()
print(f"front: {len(front)} trials, admissible (n_trades ≥ {MIN_TRADES}): {len(admissible)}")

_INTS = ("formation", "skip", "turnover_window", "rebalance_days", "max_positions")
def coerce(params):
    """Cast params back to real types (front rows store everything as object)."""
    return {k: (int(round(float(v))) if k in _INTS else v if isinstance(v, str) else float(v))
            for k, v in params.items()}

X, Y = opt.OBJECTIVES                       # (excess_cagr, tracking_error)
pool = admissible if len(admissible) else front
# ONE picking rule for the whole repo (opt.PICK_WEIGHTS, balanced), and it collapses tied
# outcomes first: `n_tied` > 1 says the pick was arbitrary among that many equal trials.
best_row = opt.pick_from_front(pool)
print(f"{int(best_row['front_distinct'])} distinct outcomes on the admissible front"
      + (f" — picked one of {int(best_row['n_tied'])} TIED trials, arbitrarily"
         if best_row["n_tied"] > 1 else ""))
picked = coerce({k: best_row[k] for k in strat.optuna["space"]})

beats = front["excess_cagr"] > opt.MIN_PICK_EXCESS
print(f"trials on the front that BEAT buy-and-hold: {int(beats.sum())} of {len(front)}"
      + ("" if best_row["beats_benchmark"] else " — none do, so the pick is flagged"))
print("picked params:", picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4)
      for m in (X, Y, "info_ratio", "sharpe", "cagr", "pct_profitable", "n_trades",
                "trades_per_year", "years")})
plots.plot_pareto(front, opt.trials_frame(study), picked=int(best_row["trial"])).show()

## 8 · Boundary check

If a picked value sits on a search-space edge, the real optimum may lie outside
the box — widen that bound and re-run. (Edge flags driven only by the thin
few-trade tail are false positives; judge against the admissible front.)

In [ ]:
display(opt.boundary_report(picked, strat.optuna["space"], front=front))

## 9 · Backtest the picked parameters

In [ ]:
res_best = bt.run_strategy(strat, panel, params=picked, settings=settings)
stats_best = bt.portfolio_stats(res_best)
for k, v in stats_best.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res_best)).show()
display(bt.symbol_stats(res_best).sort_values("total_return", ascending=False).head(10).round(3))

## 10 · Validate — is the edge real?

Everything above is in-sample. **§10a is the one that can kill the strategy
outright** — a long-only ranker earns a positive absolute Sharpe merely by being
invested, so the only question that counts is whether it beats simply owning the
universe.

In [ ]:
def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    feats = {k: v.loc[idx] for k, v in (p.features or {}).items()}
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx], features=feats)

### 10a · The hurdle: does it beat simply owning the universe?

The reference is an **equal-weight buy-and-hold** of the same names over the same
period, paying the same costs. If the strategy does not beat it, the stock
selection is adding nothing — regardless of how healthy the headline looks. This
is also what the optimizer scores (`excess_cagr` against `tracking_error`, both relative
to this benchmark).

In [ ]:
def hurdle(p, params, label):
    """Strategy vs equal-weight buy-and-hold of the same panel. The pass/fail gate."""
    st = bt.evaluate(strat, p, params, settings=settings)
    fees, slip = bt.panel_costs(settings.costs, p)
    bench_ret, _ = bt.benchmark(p, fees=fees, slippage=slip, fill=strat.fill)
    bench_cagr = bt._cagr(bench_ret, st["years"])
    ok = st["excess_cagr"] > 0 and st["info_ratio"] > 0
    verdict = ("PASS — it beats owning the universe" if ok
               else "FAIL — owning the universe was better")
    print(f"\n{label}  ({p.close.index[0].date()} -> {p.close.index[-1].date()}, "
          f"{st['years']:.2f} years, {st['trades_per_year']:.0f} trades/yr)")
    print(f"  strategy     {st['cagr']:+7.2%}/yr   sharpe {st['sharpe']:5.2f}   "
          f"n_trades {int(st['n_trades'])}")
    print(f"  buy & hold   {bench_cagr:+7.2%}/yr")
    print(f"  EXCESS       {st['excess_cagr']:+7.2%}/yr   "
          f"information ratio {st['info_ratio']:+5.2f}")
    print(f"  --> {verdict}")
    return st

hurdle(panel, picked, "TRAIN")
full_h = load_panel(engine, syms, d["bar"], d["start"], None,
                    session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(full_h, engine, d["features"], settings=settings)
holdout = slice_panel(full_h, "2025-01-01", str(full_h.close.index[-1].date()))
hurdle(holdout, picked, "HOLDOUT (2025+)")

### 10b · Costs on, then doubled — the breakeven, and the two fixed bars

Never trust a zero-cost result. Re-run the pick at 1×/2×/3× the configured costs, find the
round-trip cost (in bps) at which the Sharpe hits zero, then score it at the repo's two
fixed bars — **6/5 bps** (spread/slippage per leg, the pass bar) and **10/3 bps** — with the
benchmark re-priced at the same costs. A monthly rotation of ~half the book is ~4–5%/yr of
drag at the defaults; confirm the edge is not a cost-hurdle mirage.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

c = settings.costs
for mult in (1.0, 2.0, 3.0):
    s = with_costs(spread_bps=c.spread_bps * mult, slippage_bps=c.slippage_bps * mult,
                   commission_per_share=c.commission_per_share * mult)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    print(f"  {mult:.0f}x costs: sharpe={st['sharpe']:.3f}  return={st['total_return']:.2%}")

base_bps = c.spread_bps + c.slippage_bps
breakeven = None
for extra in np.arange(0, 60, 2.0):
    s = with_costs(spread_bps=base_bps + extra, slippage_bps=0.0)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    if st["sharpe"] <= 0:
        breakeven = base_bps + extra
        break
be = ">60" if breakeven is None else round(breakeven, 1)
print(f"\napprox breakeven half-spread: {be} bps per leg")

print("\nfixed cost bars (spread/slippage bps per leg), benchmark re-priced at the same costs:")
for spread, slip in ((6.0, 5.0), (10.0, 3.0)):
    s = with_costs(spread_bps=spread, slippage_bps=slip)
    st = bt.evaluate(strat, panel, picked, settings=s)
    print(f"  {spread:>4.0f}/{slip:<3.0f} bps: excess_cagr {st['excess_cagr']:+.2%}/yr   "
          f"info_ratio {st['info_ratio']:+.2f}   n_trades {int(st['n_trades'])}")

### 10c · Sub-period consistency

One number over four years can hide a strategy that only worked in one regime.
Score the pick on each calendar year and read the table, not a pass/fail.

In [ ]:
years = []
metrics = ("total_return", "excess_cagr", "sharpe", "pct_profitable", "n_trades")
for y in range(2020, 2025):   # 2020 is the half-year after the ranking window
    sub = slice_panel(panel, f"{y}-01-01", f"{y}-12-31")
    st = bt.evaluate(strat, sub, picked, settings=settings)
    years.append({"year": y, **{m: round(float(st[m]), 3) for m in metrics}})
display(pd.DataFrame(years).set_index("year"))

### 10d · Untouched holdout (open once)

2025+ was never seen by the config, the sweep, or the optimizer. Score the pick on
it exactly once. If it disappoints, that is information — do not tune against it.

In [ ]:
st = bt.portfolio_stats(bt.run_strategy(strat, holdout, params=picked, settings=settings))
print(f"HOLDOUT {holdout.close.index[0].date()} → {holdout.close.index[-1].date()} "
      f"({holdout.close.shape[0]} bars):")
for k, v in st.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(bt.run_strategy(strat, holdout, params=picked,
                                                  settings=settings))).show()

## Read the result honestly

- **§4 decides whether there is anything to trade**; §10a decides whether the *strategy*
  harvests it after costs against owning the universe. Both must hold, on train, before
  the holdout is opened once.
- Judge breadth on **`pct_profitable`**, and read **`n_trades`** next to every Sharpe. A
  10-slot momentum book is lumpy by construction.
- The universe is **survivor-only** (the database retains no delisted names), so absolute
  returns and drawdowns are flattered; the benchmark-relative numbers are the honest ones.
- Momentum has crashes. Read §10c for 2022 and early 2023 before believing a headline;
  `retrim_k` caps single names, nothing caps the factor.
- `turnover_source` in §4 must read `shares`. If it reads `dollar_volume`, the fundamentals
  sleeve is thin for this universe and the strategy is testing a *liquidity* sort, not a
  turnover one — ingest `fund_shares` for `Liquid500` first.
- Expected magnitude, honestly: the long leg's excess over equal-weight is plausibly
  +5–15%/yr in trending regimes and negative in reversal regimes. Not a lottery ticket.